# PPTX-dbx — deck build notebook (template)

Import this notebook into Databricks (Workspace → Import → File). Cells are ordered: install → restart → paths → three-role build → QA → deliver.

Replace every `<<...>>` placeholder before running. The skill files (this skill's `scripts/`, `agents/`, `references/`) are assumed to live at `SKILL_DIR` — typically a Databricks Repo path.

## 1. Install Python packages

Run once per session. Since DBR 13.0, `%pip install` does NOT auto-restart the kernel — the explicit `dbutils.library.restartPython()` in the next cell is required.

In [ ]:
%pip install python-pptx "markitdown[pptx]" Pillow defusedxml
# Optional, only when LibreOffice is unavailable (serverless compute):
# %pip install aspose-slides

In [ ]:
dbutils.library.restartPython()

## 2. Paths and imports

`SKILL_DIR` points to where the pptx-dbx skill lives (typically a Databricks Repo). `OUT_DIR` is the Volume directory where the deck and PDF will land — confirm your principal can write there before running.

In [ ]:
import os, subprocess, urllib.parse
from pathlib import Path

SKILL_DIR = "/Workspace/Repos/<<your-user>>/pptx-dbx"
OUT_DIR   = "/Volumes/<<catalog>>/<<schema>>/<<volume>>/decks"
DECK_NAME = "<<deck_name>>"   # no extension

os.makedirs(OUT_DIR, exist_ok=True)

SPEC_PATH = f"/tmp/{DECK_NAME}.spec.md"
PY_PATH   = f"{OUT_DIR}/{DECK_NAME}.py"
PPTX_PATH = f"{OUT_DIR}/{DECK_NAME}.pptx"
PDF_PATH  = f"{OUT_DIR}/{DECK_NAME}.pdf"

print("SKILL_DIR :", SKILL_DIR)
print("SPEC_PATH :", SPEC_PATH)
print("PPTX_PATH :", PPTX_PATH)

## 2b. Subagent dispatcher (Databricks Model Serving)

**Why not call another Genie Code from here?** Genie Code is a UI assistant — it lives in the notebook chat panel and `Cmd+I` cell prompt, and Databricks does NOT expose a public SDK or REST API for one Genie Code session to spawn another. (Don't confuse it with **AI/BI Genie**, which is a different product for SQL/table Q&A and *does* have `WorkspaceClient().genie.start_conversation_and_wait(...)` — but that one can't run python-pptx.)

To get a programmatically-isolated subagent inside a notebook cell, set `DBX_ENDPOINT` to a chat-completions endpoint already provisioned in your workspace (e.g. `databricks-claude-sonnet-4`). Each `dispatch_subagent(...)` call ships the role's `agents/<role>.md` as the system prompt and only the role's allowed inputs as the user message — no memory of prior cells.

**Fallback when `DBX_ENDPOINT` is empty — single-agent mode:** the dispatcher prints the role brief + inputs and returns `None`. The Genie Code session driving the notebook then plays the role itself, inline, and writes the artifact (spec.md / .pptx / review.md) to disk. The role cell gates on the artifact's existence and continues. This is structurally weaker than true multi-agent (same brain plays every role) but preserves the spec-as-contract discipline. See SKILL.md → "Fallback: single-agent mode" for the discipline rules.

The dispatcher writes every model transcript to `/tmp/<deck>.<role>.log` (multi-agent only) so you can verify the role really ran with a clean context.

In [ ]:
"""Subagent dispatcher — one Databricks Model Serving call per role.

Set DBX_ENDPOINT to a chat-completions endpoint provisioned in your workspace
(e.g. "databricks-claude-sonnet-4") to enable the multi-agent workflow.

If DBX_ENDPOINT is empty, the notebook falls back to SINGLE-AGENT mode: the
Genie Code session driving the notebook plays all three roles serially in the
role cells (§3, §4, §7), re-reading inputs from disk between roles to simulate
context isolation. This is weaker than true multi-agent (same brain plays all
roles) but preserves the spec-as-contract discipline. See SKILL.md for the
fallback rules.
"""

DBX_ENDPOINT = ""   # <-- set to e.g. "databricks-claude-sonnet-4" for multi-agent

MULTI_AGENT_ENABLED = bool(DBX_ENDPOINT)

def _read_role_brief(role: str) -> str:
    return Path(f"{SKILL_DIR}/agents/{role}.md").read_text()

def dispatch_subagent(role: str, user_message: str, max_tokens: int = 16000) -> str | None:
    """
    Multi-agent mode: run the role via Databricks Model Serving and return its output.
    Single-agent mode (DBX_ENDPOINT empty): print the role brief + inputs and return None;
    the calling cell must then have Genie Code (or a human) act as that role inline.
    """
    role_file = role.replace("pptx-dbx-", "")
    system = _read_role_brief(role_file)

    if not MULTI_AGENT_ENABLED:
        print(f"\n{'='*70}\n[SINGLE-AGENT MODE] You are now playing the role: {role}\n{'='*70}")
        print(f"--- ROLE BRIEF (agents/{role_file}.md) ---")
        print(system)
        print(f"\n--- INPUTS YOU MAY USE (and ONLY these) ---")
        print(user_message)
        print(f"\n--- INSTRUCTIONS ---")
        print(f"Forget reasoning from prior cells. Re-read the role brief and inputs above.")
        print(f"Act as {role} and produce the role's required output.")
        print(f"When done, the next cell will gate on the artifact existing on disk.\n")
        return None

    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    resp = w.serving_endpoints.query(
        name=DBX_ENDPOINT,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user_message},
        ],
        max_tokens=max_tokens,
    )
    out = resp.choices[0].message.content

    log_path = f"/tmp/{DECK_NAME}.{role_file}.log"
    Path(log_path).write_text(
        f"=== ROLE: {role} ===\n=== ENDPOINT: {DBX_ENDPOINT} ===\n"
        f"=== SYSTEM ({len(system)} chars from agents/{role_file}.md) ===\n"
        f"=== USER MESSAGE ===\n{user_message}\n\n=== MODEL OUTPUT ===\n{out}\n"
    )
    print(f"[{role}] returned {len(out)} chars — transcript: {log_path}")
    return out

if MULTI_AGENT_ENABLED:
    print(f"MULTI-AGENT ENABLED. Endpoint: {DBX_ENDPOINT}")
else:
    print("MULTI-AGENT DISABLED — falling back to SINGLE-AGENT mode.")
    print("Genie Code (or a human) plays each role inline in cells 3, 4, 7.")
    print("Set DBX_ENDPOINT in this cell to switch to true multi-agent.")

## 3. Content-creator role (cell = subagent)

**This cell IS the content-creator subagent.** Re-read the role brief from disk; act strictly as that role; do not carry reasoning from earlier cells.

- Inputs allowed: `USER_INTENT`, `references/style.md`.
- Output: `SPEC_PATH` — the contract every later cell grades against.
- Forbidden: writing code, touching `.pptx`, peeking at the python-pptx reference.

If you are a human running this notebook, paste `agents/content-creator.md` into Genie Code's chat box for this cell and let it produce the spec.

In [ ]:
USER_INTENT = "<<one paragraph: who reads this deck, what action/belief you want afterward>>"

# ---- Subagent call: content-creator ----
spec_md = dispatch_subagent(
    role="pptx-dbx-content-creator",
    user_message=(
        f"User intent:\n{USER_INTENT}\n\n"
        f"Style reference (read and pick from):\n{Path(SKILL_DIR + '/references/style.md').read_text()}\n\n"
        f"Required output path for the spec: {SPEC_PATH}\n"
        f"Required output path for the deck (record this in the spec): {PPTX_PATH}\n\n"
        f"Write the spec.md content. Output ONLY the spec markdown — no preamble."
    ),
)

if spec_md is not None:
    # Multi-agent mode: model returned the spec, persist it.
    Path(SPEC_PATH).write_text(spec_md)
else:
    # Single-agent mode: Genie Code (or human) writes SPEC_PATH directly per the printed brief above.
    print(f"\n[SINGLE-AGENT] After acting as content-creator, write the spec to: {SPEC_PATH}")

assert Path(SPEC_PATH).exists() and Path(SPEC_PATH).stat().st_size > 0, \
    f"spec missing or empty: {SPEC_PATH} — content-creator role did not produce output"
print(f"\nSpec ready: {SPEC_PATH} ({Path(SPEC_PATH).stat().st_size} bytes)")
print(Path(SPEC_PATH).read_text()[:600], "...")

## 4. Code-generator role (cell = subagent)

**This cell IS the code-generator subagent.** Re-read the role brief; act strictly as that role.

- Inputs allowed: `SPEC_PATH`, `references/python-pptx.md`.
- Output: `PY_PATH` (re-runnable generator) + `PPTX_PATH` (the deck) at the Volume path the spec names.
- Forbidden: editing the spec, second-guessing contracts, judging your own output.

Every shape must get `shape.name = "<spec name>"` — without this the reviewer auto-FAILs.

In [ ]:
# ---- Subagent call: code-generator ----
generator_py = dispatch_subagent(
    role="pptx-dbx-code-generator",
    user_message=(
        f"Spec (this is the contract — match content verbatim, realize every alignment contract):\n"
        f"{Path(SPEC_PATH).read_text()}\n\n"
        f"python-pptx reference:\n{Path(SKILL_DIR + '/references/python-pptx.md').read_text()}\n\n"
        f"Required output path for the .py generator: {PY_PATH}\n"
        f"Required output path for the .pptx deck:    {PPTX_PATH}\n\n"
        f"Output ONLY the Python source for the generator — no markdown fences, no commentary. "
        f"It must, when executed, write {PPTX_PATH}."
    ),
    max_tokens=32000,
)

if generator_py is not None:
    # Multi-agent mode: persist and execute the model-written generator.
    Path(PY_PATH).write_text(generator_py)
    r = subprocess.run(["python", PY_PATH], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
        raise RuntimeError(f"generator failed — inspect {PY_PATH} and /tmp/{DECK_NAME}.code-generator.log")
else:
    # Single-agent mode: Genie Code writes PY_PATH and produces PPTX_PATH inline per the printed brief.
    print(f"\n[SINGLE-AGENT] After acting as code-generator, ensure these exist:\n  {PY_PATH}\n  {PPTX_PATH}")

assert Path(PPTX_PATH).exists(), f"deck missing: {PPTX_PATH} — code-generator did not write it"
print("\nBuilt:", PPTX_PATH, "—", os.path.getsize(PPTX_PATH), "bytes")

## 5. Structural QA — `markitdown`, `invariants.py`, `measure.py`

Pure Python; works on every Databricks compute. Always run before declaring success. Read the output — running these without reading them is not verification.

In [ ]:
def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r

run(["python", "-m", "markitdown", PPTX_PATH])

In [ ]:
run(["python", f"{SKILL_DIR}/scripts/invariants.py", PPTX_PATH])

In [ ]:
# Per-slide measure reports — read each one and write a verdict per shape.
for n in range(1, 99):
    out_path = f"/tmp/measure-slide-{n:02d}.txt"
    r = subprocess.run(
        ["python", f"{SKILL_DIR}/scripts/measure.py", PPTX_PATH, "--slide", str(n)],
        capture_output=True, text=True,
    )
    if r.returncode != 0 or "no slide" in r.stdout.lower():
        break
    Path(out_path).write_text(r.stdout)
    print(f"slide {n}: {out_path} ({len(r.stdout)} bytes)")

## 6. Visual QA (best-effort)

LibreOffice on classic compute, Aspose on serverless, otherwise skip and disclose. See `references/databricks.md` §5 for the decision matrix.

Aspose's free tier watermarks slide 1 — fine for QA inspection, do NOT ship the Aspose-rendered PDF.

In [ ]:
# Collect the structural-QA artifacts the reviewer needs (deck + spec + measure/invariants outputs).
inv_text     = subprocess.run(["python", f"{SKILL_DIR}/scripts/invariants.py", PPTX_PATH], capture_output=True, text=True).stdout
measure_text = "\n".join(Path(p).read_text() for p in sorted(Path("/tmp").glob("measure-slide-*.txt")))
markitdown   = subprocess.run(["python", "-m", "markitdown", PPTX_PATH], capture_output=True, text=True).stdout

REVIEW_PATH = f"/tmp/{DECK_NAME}.review.md"

def run_review(iteration: int = 1) -> tuple[str | None, bool]:
    """
    Run a brand-new reviewer.
      - Multi-agent: returns (report, passed). A new dispatch = a new isolated reviewer subagent.
      - Single-agent: returns (None, False). Genie Code (or human) writes the verdict to REVIEW_PATH, then re-run.
    """
    report = dispatch_subagent(
        role="pptx-dbx-slide-reviewer",
        user_message=(
            f"User intent (the only context you have about why this deck exists):\n{USER_INTENT}\n\n"
            f"Deck path: {PPTX_PATH}\n"
            f"Spec ({SPEC_PATH}):\n{Path(SPEC_PATH).read_text()}\n\n"
            f"markitdown output:\n{markitdown}\n\n"
            f"invariants.py output:\n{inv_text}\n\n"
            f"measure.py output (per slide):\n{measure_text}\n\n"
            f"Grade against the spec using the hard thresholds in your role brief. "
            f"Verdict is binary (PASS or FAIL). On FAIL, list exact fixes (shape name, current value, target value)."
        ),
        max_tokens=12000,
    )
    if report is None:
        # Single-agent: human/Genie writes REVIEW_PATH inline.
        if Path(REVIEW_PATH).exists():
            report = Path(REVIEW_PATH).read_text()
        else:
            print(f"\n[SINGLE-AGENT] After acting as reviewer, write the verdict to: {REVIEW_PATH}, then re-run this cell.")
            return None, False
    else:
        Path(REVIEW_PATH).write_text(report)
    passed = "**Verdict**: PASS" in report or "Verdict: PASS" in report
    return report, passed

report, passed = run_review(iteration=1)
if report:
    print(report)
print("=" * 60)
print("VERDICT:", "PASS" if passed else "FAIL")
print("Re-review loop: if FAIL, re-run the code-generator cell with this punch list, then re-run THIS cell.")
print("Each re-run of this cell IS a brand-new reviewer (fresh context in multi-agent mode).")

## 7. Reviewer role (cell = subagent) — fresh context

**This cell IS the reviewer subagent.** It must form an independent judgment.

- Inputs allowed: `PPTX_PATH`, `SPEC_PATH`, `USER_INTENT`.
- Forbidden: the creator's reasoning, the generator's transcript, "the generator says X is intentional", any prior reviewer report.
- Output: PASS or FAIL. On FAIL, an exact punch list (shape `name`, current value, target value).

**Re-review loop**: if FAIL, re-run cell 4 (code-generator) feeding it the punch list, then re-run THIS cell as a **brand-new** reviewer (do NOT reuse the prior reviewer's reasoning). Up to 9 iterations, then surface to the user with the last report.

In [ ]:
# Genie Code prompt for THIS cell (paste verbatim, then RUN with no other context):
#   "Read SKILL_DIR/agents/slide-reviewer.md. You have NOT seen the build.
#    Inputs:
#      deck = PPTX_PATH
#      spec = SPEC_PATH
#      user intent = <USER_INTENT>
#    Grade the deck against the spec using the hard thresholds in the role brief.
#    Return a structured report. Verdict is binary (PASS or FAIL). On FAIL, list exact fixes
#    with shape name + current value + target value. Do NOT soften the verdict."

# Persist the reviewer's last verdict on disk so the orchestrator can read it programmatically:
REVIEW_PATH = f"/tmp/{DECK_NAME}.review.md"
print("Reviewer must write its report to:", REVIEW_PATH)
print("Inputs the reviewer is allowed to see:")
print("  deck =", PPTX_PATH)
print("  spec =", SPEC_PATH)
print("  user intent =", USER_INTENT[:120], "...")

## 8. PDF sibling (optional)

Only if the user asked for PDF. Use LibreOffice — Aspose's free tier watermarks slide 1 and is unacceptable for delivery.

In [ ]:
if soffice:
    subprocess.run([
        "python", f"{SKILL_DIR}/scripts/office/soffice.py",
        "--headless", "--convert-to", "pdf", PPTX_PATH, "--outdir", OUT_DIR,
    ], check=True)
    print("PDF written:", PDF_PATH)
else:
    print("PDF skipped — LibreOffice not available on this cluster.")

## 9. Deliver to the user — download links

Print the absolute path AND emit a clickable download link via `displayHTML`. Volume files use the `/api/2.0/fs/files{path}` endpoint; Workspace files use `/files/...`. The user's browser session is authenticated, so clicking triggers a normal download — no token in the URL.

In [ ]:
def download_link(path: str, label: str | None = None) -> str:
    label = label or path.rsplit("/", 1)[-1]
    if path.startswith("/Volumes/"):
        href = "/api/2.0/fs/files" + urllib.parse.quote(path)
    elif path.startswith("/Workspace/"):
        href = "/files" + path[len("/Workspace"):]
    else:
        raise ValueError(f"Unsupported path for download link: {path}")
    return (
        f'<a href="{href}" download '
        f'style="display:inline-block;padding:10px 16px;background:#1f6feb;color:#fff;'
        f'border-radius:6px;text-decoration:none;font-weight:600;margin:4px 8px 4px 0">'
        f'⬇️ {label}</a>'
    )

links = [download_link(PPTX_PATH, f"{DECK_NAME}.pptx")]
if Path(PDF_PATH).exists():
    links.append(download_link(PDF_PATH, f"{DECK_NAME}.pdf"))

displayHTML(
    "<div style='font-family:sans-serif;font-size:14px'>"
    "<div style='margin-bottom:8px'>Deck ready. Click to download:</div>"
    + "".join(links)
    + f"<div style='margin-top:12px;color:#666;font-size:12px'>"
    f"Volume path: <code>{PPTX_PATH}</code><br>"
    f"Backup: Catalog Explorer → navigate to the Volume → right-click → Download."
    f"</div></div>"
)

print(f"Deck saved: {PPTX_PATH}")
if Path(PDF_PATH).exists():
    print(f"PDF saved : {PDF_PATH}")